# YouTube Creator Ecosystem: Growth & Monetization Analysis
**Portfolio project — YouTube Business Analyst application**

In [ ]:
!pip install google-api-python-client pandas matplotlib seaborn scipy -q

## Configuration
Paste your YouTube Data API v3 key below.

In [ ]:
import os
import time
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from googleapiclient.discovery import build
from scipy.stats import pearsonr

API_KEY = "YOUR_API_KEY_HERE"  # paste your key here

CATEGORY_IDS = ["10", "20", "24", "22", "28", "25", "17", "27", "23"]
CATEGORY_NAMES = {
    "10": "Music", "20": "Gaming", "24": "Entertainment",
    "22": "People & Blogs", "28": "Science & Tech",
    "25": "News", "17": "Sports", "27": "Education", "23": "Comedy"
}
REGIONS = ["US", "GB", "IN", "BR", "JP"]
MAX_RESULTS = 50

## Step 1 — Fetch Trending Videos

In [ ]:
def build_client():
    return build("youtube", "v3", developerKey=API_KEY)

def fetch_trending(youtube, region="US", category_id=None, max_results=50):
    params = dict(
        part="snippet,statistics,contentDetails",
        chart="mostPopular",
        regionCode=region,
        maxResults=max_results,
    )
    if category_id:
        params["videoCategoryId"] = category_id
    resp = youtube.videos().list(**params).execute()
    rows = []
    for item in resp.get("items", []):
        stats = item.get("statistics", {})
        snippet = item.get("snippet", {})
        rows.append({
            "video_id":      item["id"],
            "title":         snippet.get("title", ""),
            "channel_id":    snippet.get("channelId", ""),
            "channel_title": snippet.get("channelTitle", ""),
            "category_id":   snippet.get("categoryId", category_id),
            "published_at":  snippet.get("publishedAt", ""),
            "view_count":    int(stats.get("viewCount", 0)),
            "like_count":    int(stats.get("likeCount", 0)),
            "comment_count": int(stats.get("commentCount", 0)),
            "region":        region,
        })
    return rows

def fetch_channel_stats(youtube, channel_ids):
    stats = {}
    for i in range(0, len(channel_ids), 50):
        batch = channel_ids[i:i+50]
        resp = youtube.channels().list(part="statistics,snippet", id=",".join(batch)).execute()
        for item in resp.get("items", []):
            s = item.get("statistics", {})
            stats[item["id"]] = {
                "subscriber_count": int(s.get("subscriberCount", 0)),
                "total_views":      int(s.get("viewCount", 0)),
                "video_count":      int(s.get("videoCount", 0)),
                "country":          item.get("snippet", {}).get("country", ""),
            }
        time.sleep(0.1)
    return stats

## Step 2 — Run the Pipeline

In [ ]:
youtube = build_client()
print("Fetching trending videos...")
all_videos = []

for cat_id in CATEGORY_IDS:
    try:
        rows = fetch_trending(youtube, region="US", category_id=cat_id)
        all_videos.extend(rows)
        print(f"  ✓ {CATEGORY_NAMES.get(cat_id, cat_id)}: {len(rows)} videos")
        time.sleep(0.2)
    except Exception as e:
        print(f"  ✗ {cat_id}: {e}")

for region in REGIONS:
    try:
        rows = fetch_trending(youtube, region=region)
        all_videos.extend(rows)
        print(f"  ✓ Region {region}: {len(rows)} videos")
        time.sleep(0.2)
    except Exception as e:
        print(f"  ✗ Region {region}: {e}")

df = pd.DataFrame(all_videos).drop_duplicates(subset="video_id")
print(f"\nTotal unique videos: {len(df)}")

channel_ids = df["channel_id"].unique().tolist()
print(f"Fetching channel stats for {len(channel_ids)} channels...")
ch_stats = fetch_channel_stats(youtube, channel_ids)
ch_df = pd.DataFrame.from_dict(ch_stats, orient="index").reset_index()
ch_df.columns = ["channel_id", "subscriber_count", "total_views", "video_count", "country"]
df = df.merge(ch_df, on="channel_id", how="left")

df["engagement_rate"]    = (df["like_count"] + df["comment_count"]) / df["view_count"].replace(0, np.nan)
df["views_per_subscriber"] = df["view_count"] / df["subscriber_count"].replace(0, np.nan)
df["avg_views_per_video"] = df["total_views"] / df["video_count"].replace(0, np.nan)
df["category_name"] = df["category_id"].map(CATEGORY_NAMES).fillna("Other")
print("Done.")

## Step 3 — Summary Table

In [ ]:
us = df[df["region"] == "US"]
summary = (
    us.groupby("category_name")
    .agg(
        channels=("channel_id", "nunique"),
        median_subscribers=("subscriber_count", "median"),
        median_views=("view_count", "median"),
        median_engagement=("engagement_rate", "median"),
    )
    .sort_values("median_subscribers", ascending=False)
)
summary["median_subscribers"] = summary["median_subscribers"].apply(lambda x: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
summary["median_views"]       = summary["median_views"].apply(lambda x: f"{x/1e6:.1f}M" if x >= 1e6 else f"{x/1e3:.0f}K")
summary["median_engagement"]  = summary["median_engagement"].apply(lambda x: f"{x:.2%}")
print("CREATOR ECOSYSTEM SUMMARY — US TRENDING")
print(summary.to_string())

print("\nREGIONAL AUDIENCE ACTIVATION (views / subscriber)")
reg = df.groupby("region")["views_per_subscriber"].agg(["median", "mean"]).round(3).sort_values("median", ascending=False)
print(reg.to_string())

## Step 4 — Charts

In [ ]:
sns.set_theme(style="whitegrid", palette="muted")
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("YouTube Creator Ecosystem: Growth & Monetization Analysis", fontsize=15, fontweight="bold", y=1.01)

cat_subs = df[df["region"]=="US"].groupby("category_name")["subscriber_count"].median().sort_values(ascending=True)
cat_subs.plot(kind="barh", ax=axes[0,0], color="#4285F4")
axes[0,0].set_title("Median Subscriber Count by Category (US)", fontsize=11)
axes[0,0].set_xlabel("Median Subscribers")
axes[0,0].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x/1e6:.1f}M" if x>=1e6 else f"{x/1e3:.0f}K"))
axes[0,0].set_ylabel("")

cat_eng = df[df["region"]=="US"].groupby("category_name")["engagement_rate"].median().sort_values(ascending=True)
cat_eng.plot(kind="barh", ax=axes[0,1], color="#34A853")
axes[0,1].set_title("Median Engagement Rate by Category", fontsize=11)
axes[0,1].set_xlabel("Engagement Rate")
axes[0,1].xaxis.set_major_formatter(mticker.FuncFormatter(lambda x,_: f"{x:.2%}"))
axes[0,1].set_ylabel("")

reg_vps = df[df["region"].isin(REGIONS)].groupby("region")["views_per_subscriber"].median().sort_values(ascending=False)
reg_vps.plot(kind="bar", ax=axes[1,0], color="#FBBC04", edgecolor="none")
axes[1,0].set_title("Views per Subscriber by Region\n(audience activation signal)", fontsize=11)
axes[1,0].set_xlabel("Region")
axes[1,0].set_ylabel("Views / Subscriber")
axes[1,0].tick_params(axis="x", rotation=0)

scatter_df = df[(df["video_count"]>10)&(df["video_count"]<10000)&(df["avg_views_per_video"]>0)&(df["avg_views_per_video"]<1e8)].dropna(subset=["video_count","avg_views_per_video"])
if len(scatter_df) > 10:
    corr, pval = pearsonr(np.log1p(scatter_df["video_count"]), np.log1p(scatter_df["avg_views_per_video"]))
    axes[1,1].scatter(scatter_df["video_count"], scatter_df["avg_views_per_video"], alpha=0.4, s=20, color="#EA4335")
    axes[1,1].set_xscale("log")
    axes[1,1].set_yscale("log")
    axes[1,1].set_title(f"Upload Volume vs. Avg Views per Video\n(Pearson r={corr:.2f}, p={pval:.3f})", fontsize=11)
    axes[1,1].set_xlabel("Total Videos Uploaded (log scale)")
    axes[1,1].set_ylabel("Avg Views per Video (log scale)")

plt.tight_layout()
plt.savefig("youtube_creator_analysis.png", dpi=150, bbox_inches="tight")
plt.show()
print("Chart saved.")

## Step 5 — Export CSV for Looker Studio

In [ ]:
out = df[["video_id","title","channel_title","category_name","region",
          "published_at","view_count","like_count","comment_count",
          "subscriber_count","video_count","engagement_rate",
          "views_per_subscriber","avg_views_per_video"]].copy()
out.to_csv("youtube_creator_data.csv", index=False)
print(f"Exported {len(out)} rows to youtube_creator_data.csv")
print("Upload this CSV to Looker Studio to build your dashboard.")